In [ ]:
import mysql.connector as mc 
from mysql.connector import Error
import dotenv
import os
from datetime import datetime, timedelta

import requests
import os
import dotenv

dotenv.load_dotenv(override=True)

APIKEY = os.getenv('PUBLIC_DATA_API_KEY')
print(f"[INFO] APIKEY={APIKEY[:5]}..., encode={0}...")

DB_CONFIG = {
        'host': os.getenv('DB_HOST'),
        'port': int(os.getenv('DB_PORT', 3306)),
        'user': os.getenv('DB_USER'),
        'password': os.getenv('DB_PASSWORD'),
        'database': os.getenv('DB_DATABASE'),
        'autocommit': False,
}
print_config = {key: DB_CONFIG[key] if key != 'password' else '****' for key in DB_CONFIG}
print(f"[INFO] DB_CONFIG={print_config}")

[INFO] decode=5e8b9..., encode=0...
[INFO] DB_CONFIG={'host': 'edhweb.iptime.org', 'port': 4050, 'user': 'skn1team', 'password': '****', 'database': 'incheon', 'autocommit': False}


In [55]:
url = 'http://apis.data.go.kr/B551177/StatusOfParking/getTrackingParking'
params ={'serviceKey' : APIKEY, 'numOfRows' : '1000', 'pageNo' : '1', 'type': 'json' }
data = requests.get(url, params).json()
data

{'response': {'header': {'resultCode': '00', 'resultMsg': 'NORMAL SERVICE.'},
  'body': {'numOfRows': 1000,
   'pageNo': 1,
   'totalCount': 19,
   'items': [{'floor': 'T1 단기주차장지하1층',
     'parking': '281',
     'parkingarea': '378',
     'datetm': '20260330135736.000'},
    {'floor': 'T1 단기주차장지하2층',
     'parking': '663',
     'parkingarea': '1334',
     'datetm': '20260330135736.000'},
    {'floor': 'T1 단기주차장지상층',
     'parking': '639',
     'parkingarea': '1052',
     'datetm': '20260330135737.000'},
    {'floor': 'T1 장기 P1 주차장',
     'parking': '2150',
     'parkingarea': '2769',
     'datetm': '20260330135737.000'},
    {'floor': 'T1 장기 P1 주차타워',
     'parking': '1371',
     'parkingarea': '1379',
     'datetm': '20260330135737.000'},
    {'floor': 'T1 장기 P2 주차장',
     'parking': '1600',
     'parkingarea': '2581',
     'datetm': '20260330135737.000'},
    {'floor': 'T1 장기 P2 주차타워',
     'parking': '1368',
     'parkingarea': '1379',
     'datetm': '20260330135737.000'},
    {'flo

In [50]:
from datetime import datetime

def analyze_parking_data(res_json):
    items = res_json["response"]["body"]["items"]

    total_count = len(items)

    times = []
    long_term = set()
    short_term = set()
    floor_ids = set()

    for item in items:
        floor = item.get("floor")
        datetm = item.get("datetm")

        # 시간 파싱
        if datetm:
            dt = datetime.strptime(datetm[:14], "%Y%m%d%H%M%S")
            times.append(dt)

        # 장기 / 단기 구분
        if floor:
            floor_ids.add(floor)

            if "장기" in floor:
                long_term.add(floor)
            elif "단기" in floor:
                short_term.add(floor)

    result = {
        "total_count": total_count,
        "time_range": {
            "min": min(times).strftime("%Y-%m-%d %H:%M:%S") if times else None,
            "max": max(times).strftime("%Y-%m-%d %H:%M:%S") if times else None,
        },
        "long_term_floors": sorted(long_term),
        "short_term_floors": sorted(short_term),
        "all_floors": sorted(floor_ids),
    }

    return result
analyze_parking_data(data)

{'total_count': 19,
 'time_range': {'min': '2026-03-30 12:23:55', 'max': '2026-03-30 12:24:56'},
 'long_term_floors': ['T1 장기 P1 주차장',
  'T1 장기 P1 주차타워',
  'T1 장기 P2 주차장',
  'T1 장기 P2 주차타워',
  'T1 장기 P3 주차장',
  'T1 장기 P4 주차장',
  'T2 P1 장기주차타워',
  'T2 P2 장기주차타워',
  'T2 장기 주차장'],
 'short_term_floors': ['T1 단기주차장지상층',
  'T1 단기주차장지하1층',
  'T1 단기주차장지하2층',
  'T2 단기주차장지상1층',
  'T2 단기주차장지상2층',
  'T2 단기주차장지상3층',
  'T2 단기주차장지상4층',
  'T2 단기주차장지하M층'],
 'all_floors': ['T1 P5 예약주차장',
  'T1 단기주차장지상층',
  'T1 단기주차장지하1층',
  'T1 단기주차장지하2층',
  'T1 장기 P1 주차장',
  'T1 장기 P1 주차타워',
  'T1 장기 P2 주차장',
  'T1 장기 P2 주차타워',
  'T1 장기 P3 주차장',
  'T1 장기 P4 주차장',
  'T2 P1 장기주차타워',
  'T2 P2 장기주차타워',
  'T2 단기주차장지상1층',
  'T2 단기주차장지상2층',
  'T2 단기주차장지상3층',
  'T2 단기주차장지상4층',
  'T2 단기주차장지하M층',
  'T2 예약 주차장',
  'T2 장기 주차장']}

b'<?xml version="1.0" encoding="UTF-8" standalone="yes"?><response><header><resultCode>99</resultCode><resultMsg>NO OPENAPI SERVICE ERROR.</resultMsg></header></response>'